In [1]:
!pip install transformers torch accelerate -q


[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: pip install --upgrade pip


In [2]:
!pip install scikit-learn -q


[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: pip install --upgrade pip


In [3]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

model_name = "Qwen/Qwen2.5-1.5B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name, torch_dtype=torch.float16, device_map="auto")
print("모델 로드 완료!")

/opt/jupyter/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 338/338 [00:22<00:00, 14.97it/s]

모델 로드 완료!


In [4]:
"""
Notebook 2 (고친 버전): Qwen2.5-1.5B-Instruct 스탠스 분류 실험
수정사항:
- 평가 루프에서 dataset 전체 → dev_dataset만 사용하도록 수정
- dev_dataset을 context_conditions.json에서 직접 필터링
"""

# ============================================================
# CELL 1: 패키지 설치
# ============================================================

!pip install transformers torch accelerate scikit-learn -q


[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: pip install --upgrade pip


In [5]:
# ============================================================
# CELL 2: 임포트
# ============================================================

import json
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from sklearn.metrics import f1_score
from collections import Counter

In [6]:
# ============================================================
# CELL 3: 데이터 로드 (dev split만)
# ============================================================

with open('context_conditions.json', encoding='utf-8') as f:
    full_dataset = json.load(f)

# ★ 핵심 수정: dev split만 사용
dataset = [d for d in full_dataset if d['split'] == 'dev']

print(f"전체 샘플 수: {len(full_dataset)}")
print(f"dev 샘플 수 : {len(dataset)}")
print()
print("=== dev 레이블 분포 ===")
label_dist = Counter(d['label'] for d in dataset)
for label, count in label_dist.most_common():
    print(f"  {label:10s}: {count}개 ({count/len(dataset)*100:.1f}%)")
print()
print("=== dev 플랫폼 분포 ===")
for platform in ['twitter', 'reddit']:
    count = sum(1 for d in dataset if d['platform'] == platform)
    print(f"  {platform:10s}: {count}개")

전체 샘플 수: 6337
dev 샘플 수 : 1447

=== dev 레이블 분포 ===
  comment   : 1181개 (81.6%)
  query     : 114개 (7.9%)
  deny      : 79개 (5.5%)
  support   : 73개 (5.0%)

=== dev 플랫폼 분포 ===
  twitter   : 1021개
  reddit    : 426개


In [7]:
# ============================================================
# CELL 4: 모델 로드
# ============================================================

MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"

print(f"모델 로드 중: {MODEL_NAME}")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="auto"
)
model.eval()
print("모델 로드 완료!")
print(f"Device: {next(model.parameters()).device}")

모델 로드 중: Qwen/Qwen2.5-1.5B-Instruct


Loading weights: 100%|██████████| 338/338 [00:00<00:00, 533.62it/s]


모델 로드 완료!
Device: cuda:0


In [8]:
# ============================================================
# CELL 5: 프롬프트 + predict 함수
# ============================================================

SYSTEM_PROMPT = """You are a stance classification expert for social media discussions about rumours.

Classify the stance of the TARGET reply using exactly one of these four labels:

- support: The reply explicitly states the rumour IS true or confirmed.
  Example: "This has been confirmed." / "Yes this is real." / "Confirmed by official sources."
  NOT support: neutral reporting, sad reactions to possible news

- deny: The reply explicitly states the rumour IS false or fabricated.
  Example: "This is fake." / "No crash was reported by any official source."
  NOT deny: asking questions, expressing doubt, saying "if true"

- query: The reply asks for sources, evidence, or verification.
  Example: "Does anyone have a link?" / "Can anyone confirm this?"
  NOT query: statements that happen to mention uncertainty

- comment: Everything else. The reply does not directly address whether the rumour is true or false.
  Example: "Thoughts and prayers." / "This is the third incident." / "Terrible if true."
  When in doubt, choose comment. It is the most common label by far.

Respond with ONLY one word: support, deny, query, or comment. No explanation."""


def build_prompt(text: str) -> list:
    cleaned = text
    cleaned = cleaned.replace("[Source]",     "Rumour post:")
    cleaned = cleaned.replace("[Context]",    "Previous reply:")
    cleaned = cleaned.replace("[Misleading]", "Another reply:")
    cleaned = cleaned.replace("[Target]",     "Reply to classify:")

    user_content = (
        f"Read the following and classify the stance of the 'Reply to classify'.\n\n"
        f"{cleaned}\n\n"
        f"Stance label (support / deny / query / comment):"
    )
    return [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": user_content},
    ]


VALID_LABELS = {"support", "deny", "query", "comment"}


def predict(text: str, max_new_tokens: int = 10) -> str:
    messages   = build_prompt(text)
    text_input = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )
    input_ids = tokenizer(text_input, return_tensors="pt").input_ids.to(model.device)

    with torch.no_grad():
        output_ids = model.generate(
            input_ids,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            temperature=1.0,
            pad_token_id=tokenizer.eos_token_id,
        )

    new_tokens = output_ids[0][input_ids.shape[-1]:]
    raw_output = tokenizer.decode(new_tokens, skip_special_tokens=True).strip().lower()

    for label in VALID_LABELS:
        if label in raw_output:
            return label
    return "invalid"

In [9]:
# ============================================================
# CELL 6: predict 동작 확인
# ============================================================

test_cases = [
    ("Does anyone have a link to an official report? Can't verify this.", "query"),
    ("This is completely fake. No official sources confirm this at all.", "deny"),
    ("Confirmed by multiple official sources. Very sad news.", "support"),
    ("Thoughts and prayers for those affected by this tragedy.", "comment"),
    ("This is the third incident this month.", "comment"),
    ("Terrible if true.", "comment"),
]

print("=== predict 함수 테스트 ===")
correct = 0
for text, gold in test_cases:
    pred   = predict(text)
    status = "✓" if pred == gold else "✗"
    if pred == gold:
        correct += 1
    print(f"  {status} gold={gold:8s} pred={pred:8s} | {text[:60]}")

print(f"\n정확도: {correct}/{len(test_cases)}")

[transformers] The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


=== predict 함수 테스트 ===
  ✓ gold=query    pred=query    | Does anyone have a link to an official report? Can't verify 
  ✓ gold=deny     pred=deny     | This is completely fake. No official sources confirm this at
  ✗ gold=support  pred=deny     | Confirmed by multiple official sources. Very sad news.
  ✓ gold=comment  pred=comment  | Thoughts and prayers for those affected by this tragedy.
  ✗ gold=comment  pred=deny     | This is the third incident this month.
  ✓ gold=comment  pred=comment  | Terrible if true.

정확도: 4/6


In [10]:
# ============================================================
# CELL 7: 조건별 실험 실행 (dev split만)
# ============================================================

CONDITIONS = ["reply_only", "useful", "irrelevant", "conflicting", "mixed", "lexical"]
LABEL_MAP  = {"support": 0, "deny": 1, "query": 2, "comment": 3}


def run_condition(condition: str, data: list) -> dict:
    golds, preds  = [], []
    invalid_count = 0
    skipped_count = 0
    total         = len(data)

    for i, d in enumerate(data):
        if d[condition] is None:
            skipped_count += 1
            continue

        pred = predict(d[condition])

        if pred == "invalid":
            invalid_count += 1
            continue

        golds.append(LABEL_MAP[d["label"]])
        preds.append(LABEL_MAP[pred])

        if (i + 1) % 100 == 0:
            print(f"  [{condition}] {i+1}/{total} | 유효 {len(golds)} | invalid {invalid_count}")

    return {
        "golds":         golds,
        "preds":         preds,
        "invalid_count": invalid_count,
        "skipped_count": skipped_count,
        "n":             len(golds),
    }


def evaluate(result: dict) -> dict:
    golds = result["golds"]
    preds = result["preds"]

    if len(golds) == 0:
        return {"macro_f1": 0.0, "per_class": [0.0] * 4}

    macro_f1  = f1_score(golds, preds, average="macro",  zero_division=0)
    per_class = f1_score(golds, preds, average=None,
                         labels=[0, 1, 2, 3], zero_division=0)
    return {
        "macro_f1":  macro_f1,
        "per_class": per_class.tolist(),
    }


all_results = {}

for condition in CONDITIONS:
    print(f"\n{'='*55}")
    print(f"  조건: {condition}")
    print(f"{'='*55}")

    result  = run_condition(condition, dataset)
    metrics = evaluate(result)

    all_results[condition] = {"raw": result, "metrics": metrics}

    pc = metrics["per_class"]
    print(f"  완료 → n={result['n']} | invalid={result['invalid_count']} | skipped={result['skipped_count']}")
    print(f"  macro-F1 : {metrics['macro_f1']:.4f}")
    print(f"  support={pc[0]:.3f}  deny={pc[1]:.3f}  query={pc[2]:.3f}  comment={pc[3]:.3f}")


  조건: reply_only
  [reply_only] 100/1447 | 유효 100 | invalid 0
  [reply_only] 200/1447 | 유효 200 | invalid 0
  [reply_only] 300/1447 | 유효 300 | invalid 0
  [reply_only] 400/1447 | 유효 400 | invalid 0
  [reply_only] 500/1447 | 유효 500 | invalid 0
  [reply_only] 600/1447 | 유효 600 | invalid 0
  [reply_only] 700/1447 | 유효 700 | invalid 0
  [reply_only] 800/1447 | 유효 800 | invalid 0
  [reply_only] 900/1447 | 유효 900 | invalid 0
  [reply_only] 1000/1447 | 유효 1000 | invalid 0
  [reply_only] 1100/1447 | 유효 1100 | invalid 0
  [reply_only] 1200/1447 | 유효 1200 | invalid 0
  [reply_only] 1300/1447 | 유효 1300 | invalid 0
  [reply_only] 1400/1447 | 유효 1400 | invalid 0
  완료 → n=1447 | invalid=0 | skipped=0
  macro-F1 : 0.1644
  support=0.048  deny=0.117  query=0.399  comment=0.094

  조건: useful
  [useful] 100/1447 | 유효 100 | invalid 0
  [useful] 200/1447 | 유효 200 | invalid 0
  [useful] 300/1447 | 유효 300 | invalid 0
  [useful] 400/1447 | 유효 400 | invalid 0
  [useful] 500/1447 | 유효 500 | invalid 0
  [useful

In [11]:
# ============================================================
# CELL 9: 메인 결과 테이블
# ============================================================

print("\n" + "="*75)
print(f"{'조건':<15} {'support':>8} {'deny':>8} {'query':>8} {'comment':>8} {'macro':>8} {'n':>6}")
print("="*75)

for condition in CONDITIONS:
    m  = all_results[condition]["metrics"]
    n  = all_results[condition]["raw"]["n"]
    pc = m["per_class"]
    print(f"{condition:<15} {pc[0]:>8.3f} {pc[1]:>8.3f} {pc[2]:>8.3f} {pc[3]:>8.3f} {m['macro_f1']:>8.3f} {n:>6}")

print("="*75)


조건               support     deny    query  comment    macro      n
reply_only         0.048    0.117    0.399    0.094    0.164   1447
useful             0.025    0.126    0.453    0.087    0.173   1447
irrelevant         0.053    0.127    0.420    0.110    0.178   1447
conflicting        0.041    0.131    0.395    0.094    0.165   1244
mixed              0.042    0.129    0.378    0.055    0.151   1244
lexical            0.000    0.107    0.372    0.007    0.122   1447


In [12]:
# ============================================================
# CELL 10: Sensitivity Gap (useful 대비 하락폭)
# ============================================================

print("\n=== Sensitivity Gap (useful 대비 하락폭) ===")
baseline_macro = all_results["useful"]["metrics"]["macro_f1"]
baseline_pc    = all_results["useful"]["metrics"]["per_class"]

print(f"\n{'조건':<15} {'macro gap':>10} {'support gap':>12} {'deny gap':>10} {'query gap':>10}")
print("-"*55)

for condition in CONDITIONS:
    if condition == "useful":
        continue
    m  = all_results[condition]["metrics"]
    pc = m["per_class"]
    print(f"{condition:<15}"
          f" {baseline_macro  - m['macro_f1']:>+10.4f}"
          f" {baseline_pc[0] - pc[0]:>+12.4f}"
          f" {baseline_pc[1] - pc[1]:>+10.4f}"
          f" {baseline_pc[2] - pc[2]:>+10.4f}")


=== Sensitivity Gap (useful 대비 하락폭) ===

조건               macro gap  support gap   deny gap  query gap
-------------------------------------------------------
reply_only         +0.0084      -0.0229    +0.0089    +0.0540
irrelevant         -0.0047      -0.0280    -0.0006    +0.0326
conflicting        +0.0074      -0.0155    -0.0051    +0.0571
mixed              +0.0221      -0.0164    -0.0026    +0.0748
lexical            +0.0513      +0.0253    +0.0188    +0.0807


In [13]:
# ============================================================
# CELL 11: Minority Class 분석
# ============================================================

print("\n=== Minority Class 분석 (support + deny 평균 F1) ===")
print(f"{'조건':<15} {'support':>8} {'deny':>8} {'minority avg':>14}")
print("-"*48)

for condition in CONDITIONS:
    pc           = all_results[condition]["metrics"]["per_class"]
    minority_avg = (pc[0] + pc[1]) / 2
    print(f"{condition:<15} {pc[0]:>8.3f} {pc[1]:>8.3f} {minority_avg:>14.3f}")


=== Minority Class 분석 (support + deny 평균 F1) ===
조건               support     deny   minority avg
------------------------------------------------
reply_only         0.048    0.117          0.083
useful             0.025    0.126          0.076
irrelevant         0.053    0.127          0.090
conflicting        0.041    0.131          0.086
mixed              0.042    0.129          0.085
lexical            0.000    0.107          0.054


In [14]:
# ============================================================
# CELL 12: Invalid Rate
# ============================================================

print("\n=== Invalid Rate (파싱 실패율) ===")
print(f"{'조건':<15} {'invalid':>8} {'total':>8} {'rate':>8}")
print("-"*42)

for condition in CONDITIONS:
    raw   = all_results[condition]["raw"]
    total = raw["n"] + raw["invalid_count"]
    rate  = raw["invalid_count"] / total * 100 if total > 0 else 0
    print(f"{condition:<15} {raw['invalid_count']:>8} {total:>8} {rate:>7.1f}%")


=== Invalid Rate (파싱 실패율) ===
조건               invalid    total     rate
------------------------------------------
reply_only             0     1447     0.0%
useful                 0     1447     0.0%
irrelevant             0     1447     0.0%
conflicting            0     1244     0.0%
mixed                  0     1244     0.0%
lexical                0     1447     0.0%


In [15]:
# ============================================================
# CELL 13: Mixed depth>=2 subset 분석
# ============================================================

import os
print("\n=== Mixed 조건: depth >= 2 subset 분석 ===")

try:
    with open(os.path.expanduser('~/parsed_rumoureval.json'), encoding='utf-8') as f:
        records = json.load(f)
    id_to_depth = {r['reply_id']: r['depth'] for r in records if not r['is_source']}

    mixed_all    = [d for d in dataset if d['mixed'] is not None]
    mixed_depth2 = [d for d in mixed_all
                    if id_to_depth.get(d['reply_id'], 0) >= 2]

    print(f"  mixed 전체 샘플  : {len(mixed_all)}")
    print(f"  mixed depth>=2   : {len(mixed_depth2)}")

    if mixed_depth2:
        golds_d2, preds_d2 = [], []
        for d in mixed_depth2:
            pred = predict(d['mixed'])
            if pred in LABEL_MAP:
                golds_d2.append(LABEL_MAP[d['label']])
                preds_d2.append(LABEL_MAP[pred])

        if golds_d2:
            macro = f1_score(golds_d2, preds_d2, average='macro', zero_division=0)
            print(f"  mixed depth>=2 macro-F1: {macro:.4f}")

except FileNotFoundError:
    print("  ※ parsed_rumoureval.json 없음")


=== Mixed 조건: depth >= 2 subset 분석 ===
  mixed 전체 샘플  : 1244
  mixed depth>=2   : 224
  mixed depth>=2 macro-F1: 0.0285


In [16]:
def run_condition(condition: str, data: list, predict_fn=None) -> dict:
    if predict_fn is None:
        predict_fn = predict
    golds, preds  = [], []
    invalid_count = 0
    skipped_count = 0
    total         = len(data)
    for i, d in enumerate(data):
        if d[condition] is None:
            skipped_count += 1
            continue
        pred = predict_fn(d[condition])
        if pred == "invalid":
            invalid_count += 1
            continue
        golds.append(LABEL_MAP[d["label"]])
        preds.append(LABEL_MAP[pred])
        if (i + 1) % 100 == 0:
            print(f"  [{condition}] {i+1}/{total} | 유효 {len(golds)} | invalid {invalid_count}")
    return {
        "golds":         golds,
        "preds":         preds,
        "invalid_count": invalid_count,
        "skipped_count": skipped_count,
        "n":             len(golds),
    }

In [17]:
import os

# 0.5B 모델 로드
MODEL_NAME_05 = "Qwen/Qwen2.5-0.5B-Instruct"
print(f"모델 로드 중: {MODEL_NAME_05}")
tokenizer_05 = AutoTokenizer.from_pretrained(MODEL_NAME_05)
model_05 = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME_05,
    torch_dtype=torch.float16,
    device_map="auto"
)
model_05.eval()
print("모델 로드 완료!")

# 0.5B predict 함수
def predict_05(text: str, max_new_tokens: int = 10) -> str:
    messages   = build_prompt(text)
    text_input = tokenizer_05.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True,
    )
    input_ids = tokenizer_05(text_input, return_tensors="pt").input_ids.to(model_05.device)
    with torch.no_grad():
        output_ids = model_05.generate(
            input_ids,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            temperature=1.0,
            pad_token_id=tokenizer_05.eos_token_id,
        )
    new_tokens = output_ids[0][input_ids.shape[-1]:]
    raw_output = tokenizer_05.decode(new_tokens, skip_special_tokens=True).strip().lower()
    for label in VALID_LABELS:
        if label in raw_output:
            return label
    return "invalid"

# 0.5B 실험 실행
all_results_05 = {}
for condition in CONDITIONS:
    print(f"\n{'='*55}")
    print(f"  조건: {condition}")
    print(f"{'='*55}")
    result  = run_condition(condition, dataset, predict_fn=predict_05)
    metrics = evaluate(result)
    all_results_05[condition] = {"raw": result, "metrics": metrics}
    pc = metrics["per_class"]
    print(f"  완료 → n={result['n']} | invalid={result['invalid_count']}")
    print(f"  macro-F1 : {metrics['macro_f1']:.4f}")
    print(f"  support={pc[0]:.3f}  deny={pc[1]:.3f}  query={pc[2]:.3f}  comment={pc[3]:.3f}")

# 0.5B 결과 저장
save_data_05 = {}
for condition, data in all_results_05.items():
    save_data_05[condition] = {
        "golds":         data["raw"]["golds"],
        "preds":         data["raw"]["preds"],
        "invalid_count": data["raw"]["invalid_count"],
        "skipped_count": data["raw"]["skipped_count"],
        "n":             data["raw"]["n"],
        "macro_f1":      data["metrics"]["macro_f1"],
        "per_class_f1":  data["metrics"]["per_class"],
    }

save_path = os.path.expanduser('~/experiment_results_0.5B.json')
with open(save_path, "w", encoding="utf-8") as f:
    json.dump(save_data_05, f, ensure_ascii=False, indent=2)
print(f"저장 완료: {save_path}")

# 1.5B vs 0.5B 비교 테이블
print("\n" + "="*80)
print(f"{'조건':<15} {'1.5B macro':>10} {'0.5B macro':>10} {'차이':>8}")
print("="*80)
for condition in CONDITIONS:
    m15 = all_results[condition]["metrics"]["macro_f1"]
    m05 = all_results_05[condition]["metrics"]["macro_f1"]
    print(f"{condition:<15} {m15:>10.3f} {m05:>10.3f} {m15-m05:>+8.3f}")
print("="*80)

모델 로드 중: Qwen/Qwen2.5-0.5B-Instruct


Loading weights: 100%|██████████| 290/290 [00:06<00:00, 44.46it/s]


모델 로드 완료!

  조건: reply_only
  [reply_only] 100/1447 | 유효 100 | invalid 0
  [reply_only] 200/1447 | 유효 200 | invalid 0
  [reply_only] 300/1447 | 유효 300 | invalid 0
  [reply_only] 400/1447 | 유효 400 | invalid 0
  [reply_only] 500/1447 | 유효 500 | invalid 0
  [reply_only] 600/1447 | 유효 600 | invalid 0
  [reply_only] 700/1447 | 유효 700 | invalid 0
  [reply_only] 800/1447 | 유효 800 | invalid 0
  [reply_only] 900/1447 | 유효 900 | invalid 0
  [reply_only] 1000/1447 | 유효 1000 | invalid 0
  [reply_only] 1100/1447 | 유효 1100 | invalid 0
  [reply_only] 1200/1447 | 유효 1200 | invalid 0
  [reply_only] 1300/1447 | 유효 1300 | invalid 0
  [reply_only] 1400/1447 | 유효 1400 | invalid 0
  완료 → n=1447 | invalid=0
  macro-F1 : 0.0800
  support=0.000  deny=0.128  query=0.192  comment=0.000

  조건: useful
  [useful] 100/1447 | 유효 99 | invalid 1
  [useful] 200/1447 | 유효 199 | invalid 1
  [useful] 300/1447 | 유효 299 | invalid 1
  [useful] 400/1447 | 유효 399 | invalid 1
  [useful] 500/1447 | 유효 499 | invalid 1
  [useful] 6

PermissionError: [Errno 13] Permission denied: 'experiment_results_0.5B.json'

In [18]:
import os
save_path = os.path.expanduser('~/experiment_results_0.5B.json')
with open(save_path, "w", encoding="utf-8") as f:
    json.dump(save_data_05, f, ensure_ascii=False, indent=2)
print(f"저장 완료: {save_path}")

저장 완료: /home/ubuntu/experiment_results_0.5B.json


In [19]:
print("\n" + "="*80)
print(f"{'조건':<15} {'1.5B macro':>10} {'0.5B macro':>10} {'차이':>8}")
print("="*80)
for condition in CONDITIONS:
    m15 = all_results[condition]["metrics"]["macro_f1"]
    m05 = all_results_05[condition]["metrics"]["macro_f1"]
    print(f"{condition:<15} {m15:>10.3f} {m05:>10.3f} {m15-m05:>+8.3f}")
print("="*80)


조건              1.5B macro 0.5B macro       차이
reply_only           0.164      0.080   +0.084
useful               0.173      0.074   +0.099
irrelevant           0.178      0.070   +0.108
conflicting          0.165      0.078   +0.087
mixed                0.151      0.075   +0.076
lexical              0.122      0.082   +0.040


In [1]:
import torch
print(torch.cuda.get_device_name(0))
print(f"GPU 메모리: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")

NVIDIA L4
GPU 메모리: 22.0 GB
